In [12]:
from math import remainder

import pandas as pd
from importlib_metadata import passthrough

from src.preprocessing import train_valid_test_split
from src.evaluate import precision_k
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score


In [13]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
train, valid, test = train_valid_test_split(df)
k = int(len(valid) * 0.1)

In [14]:
#baseline 0
baseline_0_precision_k = valid["y"].eq("yes").mean()
print(baseline_0_precision_k)

0.11515777056915365


In [15]:
#baseline 1
X_train = train.drop(columns=['y', 'duration'])
y_train = train["y"]
cat_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()

model = DecisionTreeClassifier(
    criterion="entropy",
    min_samples_leaf=5,
    min_samples_split=5,
    max_depth=5,
    random_state=42,
)
encoder = OneHotEncoder()

cat_pipe = Pipeline(
    [('ohe', OneHotEncoder(handle_unknown='ignore'))]
)

preprocessor = ColumnTransformer(
    [
        ('cat', cat_pipe, cat_cols)
    ],
    remainder='passthrough',
)

pipe = Pipeline(
    [
        ('preprocessor', preprocessor),
        ("model", model),
    ]
)

In [16]:
X_valid = valid.drop(columns=['y', 'duration'])
y_valid = valid['y']
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_valid)

In [17]:
X_valid = valid[X_train.columns]
pipe.fit(X_train, y_train)

yes_idx = list(pipe.classes_).index("yes")
y_score = pipe.predict_proba(X_valid)[:, yes_idx]
prec_k = precision_k(valid["y"], y_score, k=k)
print(prec_k)
list(pipe.classes_).index('yes')

avg_precision_valid = average_precision_score(y_valid, y_score, pos_label='yes')
print(avg_precision_valid)

0.2079646017699115
0.15951634007283608


In [18]:
X_test = test[X_train.columns]
y_test = test["y"]
y_test_score = pipe.predict_proba(X_test)[:, yes_idx]
k_test = int(len(y_test) * 0.1)

prec_k_test = precision_k(y_test, y_test_score, k=k_test)
avg_precision_test = average_precision_score(y_test, y_test_score, pos_label="yes")
print(f"precision_at_10pct_test = {prec_k_test:.6f}")
print(f"average_precision_test = {avg_precision_test:.6f}")

precision_at_10pct_test = 0.530973
average_precision_test = 0.463338
